In [2]:
import requests
from bs4 import BeautifulSoup
import os
import time
from datetime import datetime

In [3]:

URL = "https://en.wikipedia.org/wiki/Lionel_Messi"

headers = {
    "User-Agent": "Aula-CPA"
}

#pastas
output_dir = "data"
html_dir = os.path.join(output_dir, "html")

os.makedirs(output_dir, exist_ok=True)

metadata = {}

#limpar html antigo
import shutil
if os.path.exists(html_dir):
    shutil.rmtree(html_dir)

os.makedirs(html_dir)

#baixar HTML principal
response = requests.get(URL, headers=headers)
response_soup = BeautifulSoup(response.text, "html.parser")

main_html_path = os.path.join(html_dir, "main.html")

with open(main_html_path, "w", encoding="utf-8") as f:
    f.write(response_soup.prettify())

metadata["main.html"] = {
    "url": URL,
    "timestamp": datetime.now()
}

print("HTML principal salvo")

#extrair links A PARTIR do HTML salvo
with open(main_html_path, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

links = set()

for a in soup.find_all("a", href=True):
    href = a["href"]

    if href.startswith("/wiki/") and ":" not in href:
        full_link = "https://en.wikipedia.org" + href
        links.add(full_link)

links = list(links)

print(f"{len(links)} links encontrados")

#baixar páginas dos links
for i, link in enumerate(links[:10]):
    try:
        r = requests.get(link, headers=headers)
        r_soup = BeautifulSoup(r.text, "html.parser")

        file_name = f"page_{i}.html"
        file_path = os.path.join(html_dir, f"page_{i}.html")

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(r_soup.prettify())

        metadata[file_name] = {
            "url": link,
            "timestamp": datetime.now()
        }

        time.sleep(1)

    except:
        continue

print("Download concluído")

HTML principal salvo
2051 links encontrados
Download concluído


In [8]:
import pandas as pd

html_dir = os.path.join(output_dir, "html")

#Sempre reinicia (evita duplicação)
data_pages = []
data_images = []

for file in os.listdir(html_dir):
    path = os.path.join(html_dir, file)

    if not os.path.isfile(path):
        continue

    with open(path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    #título
    title_tag = soup.find("h1")
    title = title_tag.text if title_tag else None

    #conteúdo principal
    content = soup.find("div", {"id": "mw-content-text"})
    first_paragraph = None

    if content:
        paragraphs = content.find_all("p", recursive=True)

        for p in paragraphs:
            text = p.get_text(strip=True)
            if text:
                first_paragraph = text
                break

    #imagens
    images = [
        "https:" + img["src"] if img["src"].startswith("//") else img["src"]
        for img in soup.find_all("img")
        if img.get("src")
    ]

    #links internos
    internal_links = [
        a["href"] for a in soup.find_all("a", href=True)
        if a["href"].startswith("/wiki/")
    ]

    link_principal = metadata[file]["url"]
    timestamp = metadata[file]["timestamp"]

    #salva uma vez só
    data_pages.append({
        "title": title,
        "first_paragraph": first_paragraph,
        "link_principal": link_principal,
        "internal_links": ["https://en.wikipedia.org" + link for link in internal_links],
        "timestamp": timestamp
    })

    #imagens
    for img in images:
        data_images.append({
            "image_url": img
        })

In [9]:
df_pages = pd.DataFrame(data_pages)
df_images = pd.DataFrame(data_images)

df_pages.to_csv("data/pages.csv",sep=";", index=False)
df_images.to_csv("data/images.csv", sep=";", index=False)

In [10]:
df_pages.tail(20)

,title,first_paragraph,link_principal,internal_links,timestamp
0,\n\n Lionel Messi\n \n,"Lionel Andrés""Leo""Messi[note 1](born 24 June 1...",https://en.wikipedia.org/wiki/Lionel_Messi,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 15:47:03.671797
1,\n\n Marco Etcheverry\n \n,Marco Antonio Etcheverry Vargas(born 26 Septem...,https://en.wikipedia.org/wiki/Marco_Etcheverry,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 15:47:06.168141
2,\n\n 2005 FIFA Club World Championship...,The2005 FIFA Club World Championship(officiall...,https://en.wikipedia.org/wiki/2005_FIFA_Club_W...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 15:47:07.421384
3,\n\n 1968–69 La Liga\n \n,The1968–69La Ligawas the 38th season since its...,https://en.wikipedia.org/wiki/1968%E2%80%9369_...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 15:47:09.434265
4,\n\n 2020–21 La Liga\n \n,"The2020–21 La Ligaseason, also known asLa Liga...",https://en.wikipedia.org/wiki/2020%E2%80%9321_...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 15:47:11.139346
5,\n\n 1979 FIFA World Youth Championshi...,"The1979 FIFA World Youth Championship, the sec...",https://en.wikipedia.org/wiki/1979_FIFA_World_...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 15:47:13.069412
6,\n\n 2004–05 Segunda División B\n ...,Theseason 2004–05ofSegunda División Bof Spanis...,https://en.wikipedia.org/wiki/2004%E2%80%9305_...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 15:47:15.367734
7,\n\n Maurice Greene (sprinter)\n ...,"Maurice Greene(born July 23, 1974) is an Ameri...",https://en.wikipedia.org/wiki/Maurice_Greene_(...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 15:47:18.017238
8,\n\n 1991–92 European Cup\n \n,The1991–92 European Cupwas the 37th season of ...,https://en.wikipedia.org/wiki/1991%E2%80%9392_...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 15:47:19.536085
9,\n\n Paris Saint-Germain FC\n \n,"Paris Saint-Germain Football Club, commonly re...",https://en.wikipedia.org/wiki/Paris_Saint-Germ...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-08 15:47:21.238152
